In [1]:
%pip install mediapipe==0.10.14
%pip install protobuf==5.29.5

^C


In [ ]:
import os
import shutil
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np

from google.colab.patches import cv2_imshow
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_pose = mp.solutions.pose

In [ ]:
#Displaying the image with pose landmarks

image_file = 'image url'
## Setup mediapipe instance
with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        # Recolor image to RGB
        image = cv2.imread(image_file)
        image_height, image_width, _ = image.shape
        # Convert the BGR image to RGB before processing.
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark
            print(landmarks)
        except:
            pass


        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                                mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                                 )

        cv2_imshow(image)

In [ ]:
Exercise_keypoints_dict = {
    'barbell biceps curl': ['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'bench press':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'chest fly machine':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'deadlift':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW','LEFT_WRIST', 'RIGHT_WRIST','LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE'],
    'decline bench press':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'hammer curl':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'hip thrust':['RIGHT_ELBOW','LEFT_WRIST', 'RIGHT_WRIST','LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE'],
    'incline bench press':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'lat pulldown':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'lateral raises':['LEFT_SHOULDER', 'RIGHT_SHOULDER'],
    'leg extension':['LEFT_KNEE', 'RIGHT_KNEE'],
    'leg raises':['LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE'],
    'plank':['LEFT_SHOULDER', 'RIGHT_SHOULDER','LEFT_WRIST', 'RIGHT_WRIST','LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE'],
    'pull up':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'push up':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'romanian deadlift':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW','LEFT_WRIST', 'RIGHT_WRIST','LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE'],
    'russian twist':['LEFT_WRIST', 'RIGHT_WRIST','LEFT_HIP', 'RIGHT_HIP'],
    'shoulder press':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'squat':['LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE','LEFT_ANKLE', 'RIGHT_ANKLE'],
    't bar row':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'tricep dips':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW'],
    'tricep pushdown':['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW']
}

In [ ]:
for folder in os.listdir('image directory'):
  print(folder)

In [ ]:
output_path = 'output directory'
for folder in os.listdir('image directory'):
  visibilty_threshold = 0.5
  keypoint_names= Exercise_keypoints_dict[folder]
  image_count = 0
  visible_images = []
  for image in os.listdir('image directory/' + folder):
    image_file = 'image directory/' + folder + '/' + image
    img = cv2.imread(image_file)
    flipped_img = cv2.flip(img, 1)
    two_images = [img, flipped_img]
    image_count += 2
    for image_cv in two_images:
      with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
          image_height, image_width, _ = image_cv.shape
          # Convert the BGR image to RGB before processing.
          results = pose.process(cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB))
          image_cv.flags.writeable = True
          image_cv = cv2.cvtColor(image_cv, cv2.COLOR_RGB2BGR)

          # Extract landmarks
          try:
              visibilty_list = []
              landmarks = results.pose_landmarks.landmark
              for keypoint in keypoint_names:
                landmark_visibilty = landmarks[pose_landmarks[keypoint].value].visibility
                visibilty_list.append(landmark_visibilty)

              visibility = all(x > visibilty_threshold for x in visibilty_list)
              if visibility:
                path_and_name = (image_file,image)
                visible_images.append(path_and_name)
          except Exception as e:
              print(f'Image {image} failed with error: {e}')
              pass
  print(f'Image count for {folder}: {image_count}')
  print(f'Visible image count for {folder}: {len(visible_images)}')

  if len(visible_images) > 200:
    for image in visible_images:
      image_path, image_name = image
      destination_folder = f'{output_path}/{folder}/'
      os.makedirs(destination_folder, exist_ok=True)
      if os.path.exists(image_path):
          shutil.copy(image_path, destination_folder)
          print(f"Copied: {image_path} -> {destination_folder}")
      else:
          print(f"File not found: {image_path}")

    print(f"Image copying completed for folder {folder}")

In [ ]:
import os
import random
import pandas as pd

# Define root directory where the image folders are located
root_dir = "/content/drive/MyDrive/MSc in DS & AI/research/pre-processed data-3/"  # Change this to your actual root folder

# Initialize lists to store image paths and corresponding folder names
train_data = []
test_data = []

# Iterate through each folder in the root directory
for folder in os.listdir(root_dir):
    folder_path = os.path.join(root_dir, folder)
    if os.path.isdir(folder_path):  # Ensure it's a directory
        images = [os.path.join(folder, img) for img in os.listdir(folder_path) if img.endswith(('png', 'jpg', 'jpeg'))]
        num_images = len(images)

        if 250 <= num_images <= 550:
            test_size = int(0.1 * num_images)  # 10% test, 90% train
            random.shuffle(images)
            test_data.extend([(img, folder) for img in images[:test_size]])
            train_data.extend([(img, folder) for img in images[test_size:]])
        elif num_images > 550:
            sampled_images = random.sample(images, 550)  # Randomly select 550 images
            test_size = int(0.1 * 400)  # 10% test, 90% train from selected 550
            random.shuffle(sampled_images)
            test_data.extend([(img, folder) for img in sampled_images[:test_size]])
            train_data.extend([(img, folder) for img in sampled_images[test_size:]])

# Create dataframes
train_df = pd.DataFrame(train_data, columns=["image_path", "folder_name"])
test_df = pd.DataFrame(test_data, columns=["image_path", "folder_name"])

In [ ]:
train_df.to_csv("path to file/train_data.csv", index=False)
test_df.to_csv("path to file/test_data.csv", index=False)